In [ ]:
!pip -q install tqdm

import os, json, re, random, math, csv
from tqdm import tqdm


In [ ]:
!pip -q install openai

In [ ]:
os.environ["OPENAI_API_KEY"] = "Your OPENAI api key"

In [ ]:
import collections, time, json

In [ ]:
from pathlib import Path
from typing import List, Tuple, Optional

In [ ]:
from collections import Counter, defaultdict
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Hello"}]
)

print(resp.choices[0].message)

ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [ ]:
INPUT_PATH = Path("/content/drive/MyDrive/data-dass/raw/de.txt")          # Find de sentences with 10 word length.
OUT_PATH   = Path("/content/drive/MyDrive/data-dass/ablation/de_len10_100.txt")
TARGET_N   = 100
SEED       = 42

random.seed(SEED)

# word here: only consist of letters or nums, ignore punctuations.
WORD_RE = re.compile(r"[0-9A-Za-zÄÖÜäöüß]+(?:-[0-9A-Za-zÄÖÜäöüß]+)*")

def word_count_no_punct(s: str) -> int:
    return len(WORD_RE.findall(s))

def normalize_for_dedup(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s

candidates = []
seen = set()

with INPUT_PATH.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        s = normalize_for_dedup(line)
        if not s:
            continue
        if s in seen:
            continue
        if word_count_no_punct(s) == 10:
            seen.add(s)
            candidates.append(s)

print(f"Found {len(candidates)} unique sentences with exactly 10 words.")

if len(candidates) < TARGET_N:
    print(f"[WARN] Not enough sentences to sample {TARGET_N}. Will output all {len(candidates)}.")
    sample = candidates
else:
    sample = random.sample(candidates, TARGET_N)

OUT_PATH.write_text("\n".join(sample) + "\n", encoding="utf-8")
print(f"Saved -> {OUT_PATH.resolve()}")

print("\nPreview:")
for i, s in enumerate(sample[:10], 1):
    print(f"{i:02d}. {s}")

Found 32927 unique sentences with exactly 10 words.
Saved -> /content/drive/MyDrive/data-dass/ablation/de_len10_100.txt

Preview:
01. Kannst du die Entfernung von der Erde zum Mond berechnen?
02. Wenn ich einen Hund hätte, würde ich ihn Tom nennen.
03. Das weiß ich nicht, ich habe es noch nicht probiert.
04. Es hielten sich zwei berühmte Künstler in dem Hotel auf.
05. Ich habe nicht erwartet, dass von dir so etwas kommt.
06. Der Champion in der Fliegengewichtsklasse kämpft gegen einen ernstzunehmenden Herausforderer.
07. Meine Schwester kann nicht gut kochen und ich auch nicht.
08. Ich verstehe nicht, warum er nicht die Wahrheit gesagt hat.
09. Man will immer das haben, was man nicht bekommen kann.
10. Ich weiß nicht, wie der nachts noch ruhig schlafen kann.


In [ ]:
INPUT_FILE = "/content/drive/MyDrive/data-dass/ablation/de_len10_100.txt"
MODEL = "gpt-4o"
MAX_RETRIES = 5
SLEEP_BETWEEN_CALLS = 0.8

OUT_FILES = {
    20: "/content/drive/MyDrive/data-dass/ablation/bar_20.txt",
    40: "/content/drive/MyDrive/data-dass/ablation/bar_40.txt",
    60: "/content/drive/MyDrive/data-dass/ablation/bar_60.txt",
    80: "/content/drive/MyDrive/data-dass/ablation/bar_80.txt",
    100: "/content/drive/MyDrive/data-dass/ablation/bar_100.txt",
}

REPLACE_WORDS = {
    20: 2,
    40: 4,
    60: 6,
    80: 8,
    100: 10,
}

WORD_RE = re.compile(r"[0-9A-Za-zÄÖÜäöüß]+(?:-[0-9A-Za-zÄÖÜäöüß]+)*")

# split sentences into：word token / non-word token
TOKEN_RE = re.compile(r"[0-9A-Za-zÄÖÜäöüß]+(?:-[0-9A-Za-zÄÖÜäöüß]+)*|[^0-9A-Za-zÄÖÜäöüß]+")

In [ ]:
def normalize_spaces_keep_line(s: str) -> str:
    return (s or "").strip()

def split_tokens_keep_punct(text: str) -> List[str]:
    return TOKEN_RE.findall(text)

def is_word_token(tok: str) -> bool:
    return bool(WORD_RE.fullmatch(tok))

def extract_word_tokens(tokens: List[str]) -> List[str]:
    return [t for t in tokens if is_word_token(t)]

def validate_10_word_sentence(line: str) -> Tuple[List[str], List[str]]:
    """
    return:
      tokens: split tokens from the line(include non-word tokens)
      words: 10 word tokens from the line
    """
    tokens = split_tokens_keep_punct(line)
    words = extract_word_tokens(tokens)
    if len(words) != 10:
        raise ValueError(f"This sentence is not 10 words: {line!r} -> {len(words)} words")
    return tokens, words

def parse_json_from_text(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return json.loads(text)

def valid_single_token(tok: str) -> bool:
    if not isinstance(tok, str):
        return False
    tok = tok.strip()
    if not tok:
        return False
    if re.search(r"\s", tok):
        return False
    if not WORD_RE.fullmatch(tok):
        return False
    return True

In [ ]:
def call_gpt_json(prompt: str, model: str = MODEL) -> dict:
    resp = client.responses.create(
        model=model,
        input=prompt,
        temperature=0,
    )
    return parse_json_from_text(resp.output_text)

def build_translation_prompt(source_words: List[str]) -> str:
    return f"""
You are translating German words into Bavarian (Bairisch).

Task:
Translate the following 10 German source words into exactly 10 Bavarian words, in the SAME order.

Rules:
- Return exactly 10 output words.
- Each German word must map to exactly ONE Bavarian word.
- No output word may contain spaces.
- Do not add punctuation. Do not change the order.
- Every output word must be different from the corresponding German source word (case-insensitive).
- Return ONLY a JSON object in this exact format:
{{
  "translations": ["w1", "w2", "w3", "w4", "w5", "w6", "w7", "w8", "w9", "w10"]
}}

Source words:
{json.dumps(source_words, ensure_ascii=False)}
""".strip()

In [ ]:
def validate_translations(source_words: List[str], translations: List[str]) -> bool:
    if not isinstance(translations, list) or len(translations) != 10:
        return False
    for src, tgt in zip(source_words, translations):
        if not valid_single_token(tgt):
            return False
        if src.strip().lower() == tgt.strip().lower():
            return False
    return True

def get_bavarian_word_translations(source_words: List[str], max_retries: int = MAX_RETRIES) -> Optional[List[str]]:
    prompt = build_translation_prompt(source_words)

    for _ in range(max_retries):
        try:
            data = call_gpt_json(prompt)
            translations = data["translations"]
            if validate_translations(source_words, translations):
                return translations
        except Exception:
            pass
        time.sleep(0.6)

    return None


In [ ]:
def apply_first_k_word_replacements(tokens: List[str], replacement_words: List[str], k: int) -> str:
    out = []
    word_idx = 0

    for tok in tokens:
        if is_word_token(tok):
            if word_idx < k:
                out.append(replacement_words[word_idx])
            else:
                out.append(tok)
            word_idx += 1
        else:
            out.append(tok)

    return "".join(out)

In [ ]:
input_path = Path(INPUT_FILE)
if not input_path.exists():
    raise FileNotFoundError(f"Files not found：{INPUT_FILE}")

raw_lines = input_path.read_text(encoding="utf-8").splitlines()
raw_lines = [normalize_spaces_keep_line(x) for x in raw_lines if normalize_spaces_keep_line(x)]

print(f"read {len(raw_lines)} lines in total.")

parsed = []
bad_lines = []

for i, line in enumerate(raw_lines, start=1):
    try:
        tokens, words = validate_10_word_sentence(line)
        parsed.append({
            "line_no": i,
            "raw": line,
            "tokens": tokens,
            "words": words,
        })
    except Exception:
        bad_lines.append({
            "line_no": i,
            "raw": line,
        })

print(f"applicable sentences(exact 10 words）: {len(parsed)}")
print(f"ignore（not 10 words）: {len(bad_lines)}")

if not parsed:
    raise ValueError("No applicable sentences")

In [ ]:
outputs = {20: [], 40: [], 60: [], 80: [], 100: []}
failed = []

for idx, item in enumerate(parsed, start=1):
    source_words = item["words"]
    tokens = item["tokens"]

    translations = get_bavarian_word_translations(source_words)

    if translations is None:
        failed.append({
            "line_no": item["line_no"],
            "source": item["raw"],
            "words": source_words,
        })
        print(f"[FAIL] line {item['line_no']}")
        continue

    # build 20/40/60/80/100% replacement
    for pct, k in REPLACE_WORDS.items():
        new_sent = apply_first_k_word_replacements(tokens, translations, k)
        outputs[pct].append(new_sent)

    if idx % 10 == 0 or idx == len(parsed):
        print(f"Processed {idx}/{len(parsed)}")

    time.sleep(SLEEP_BETWEEN_CALLS)


In [ ]:
for pct, path in OUT_FILES.items():
    Path(path).write_text("\n".join(outputs[pct]) + ("\n" if outputs[pct] else ""), encoding="utf-8")
    print(f"Saved {path}: {len(outputs[pct])} lines")

if failed:
    Path("/content/drive/MyDrive/data-dass/ablation/bar_failed.json").write_text(
        json.dumps(failed, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"Saved bar_failed.json with {len(failed)} failed lines")
else:
    print("No failed lines.")

if bad_lines:
    Path("/content/drive/MyDrive/data-dass/ablation/bar_skipped_non10.json").write_text(
        json.dumps(bad_lines, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"Saved bar_skipped_non10.json with {len(bad_lines)} skipped lines")


for pct in [20, 40, 60, 80, 100]:
    print(f"\n=== Preview bar_{pct}.txt ===")
    for s in outputs[pct][:3]:
        print(s)

Saved /content/drive/MyDrive/data-dass/ablation/bar_20.txt: 39 lines
Saved /content/drive/MyDrive/data-dass/ablation/bar_40.txt: 39 lines
Saved /content/drive/MyDrive/data-dass/ablation/bar_60.txt: 39 lines
Saved /content/drive/MyDrive/data-dass/ablation/bar_80.txt: 39 lines
Saved /content/drive/MyDrive/data-dass/ablation/bar_100.txt: 39 lines
Saved bar_failed.json with 61 failed lines

=== Preview bar_20.txt ===
Kansd di die Entfernung von der Erde zum Mond berechnen?
Des woaß ich nicht, ich habe es noch nicht probiert.
Da Meister in der Fliegengewichtsklasse kämpft gegen einen ernstzunehmenden Herausforderer.

=== Preview bar_40.txt ===
Kansd di d Entfern von der Erde zum Mond berechnen?
Des woaß i ned, ich habe es noch nicht probiert.
Da Meister im da Fliegengewichtsklasse kämpft gegen einen ernstzunehmenden Herausforderer.

=== Preview bar_60.txt ===
Kansd di d Entfern vo da Erde zum Mond berechnen?
Des woaß i ned, i hob es noch nicht probiert.
Da Meister im da Fliegengewicht focht

In [ ]:
INPUT_FILE = Path("/content/drive/MyDrive/data-dass/ablation/de_len10_100.txt")
FAILED_FILE = Path("/content/drive/MyDrive/data-dass/ablation/bar_failed.json")
OUTPUT_FILE = Path("/content/drive/MyDrive/data-dass/ablation/de_len10_100_filtered.txt")


lines = INPUT_FILE.read_text(encoding="utf-8").splitlines()

failed = json.loads(FAILED_FILE.read_text(encoding="utf-8"))

failed_line_nos = set()
failed_sources = set()

for item in failed:
    if "line_no" in item and item["line_no"] is not None:
        failed_line_nos.add(int(item["line_no"]))
    if "source" in item and item["source"]:
        failed_sources.add(item["source"].strip())

filtered_lines = []
removed = []

for i, line in enumerate(lines, start=1):
    line_strip = line.strip()

    remove_this = False

    if i in failed_line_nos:
        remove_this = True

    elif line_strip in failed_sources:
        remove_this = True

    if remove_this:
        removed.append((i, line))
    else:
        filtered_lines.append(line)

OUTPUT_FILE.write_text("\n".join(filtered_lines) + ("\n" if filtered_lines else ""), encoding="utf-8")

print(f"original line: {len(lines)}")
print(f"deleted line: {len(removed)}")
print(f"rest line: {len(filtered_lines)}")
print(f"save to: {OUTPUT_FILE.resolve()}")


print("\n deleted line：")
for x in removed[:10]:
    print(x)

In [ ]:
DE_FILE = Path("/content/drive/MyDrive/data-dass/ablation/de_len10_100_filtered.txt")

BAR_FILES = {
    20: Path("/content/drive/MyDrive/data-dass/ablation/bar_20.txt"),
    40: Path("/content/drive/MyDrive/data-dass/ablation/bar_40.txt"),
    60: Path("/content/drive/MyDrive/data-dass/ablation/bar_60.txt"),
    80: Path("/content/drive/MyDrive/data-dass/ablation/bar_80.txt"),
    100: Path("/content/drive/MyDrive/data-dass/ablation/bar_100.txt"),
}

OUT_DIR = Path("/content/drive/MyDrive/data-dass/ablation/aligned-1")  # 改成你想保存的目录，比如 Path("/content/my_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def read_lines(p: Path):
    return [ln.rstrip("\n") for ln in p.read_text(encoding="utf-8").splitlines()]

de_lines = read_lines(DE_FILE)
print(f"DE lines: {len(de_lines)}")

for pct, bar_path in BAR_FILES.items():
    bar_lines = read_lines(bar_path)
    print(f"BAR {pct} lines: {len(bar_lines)}")

    n = min(len(de_lines), len(bar_lines))
    if len(de_lines) != len(bar_lines):
        print(f"[WARN] {pct}% 行数不一致：DE={len(de_lines)} BAR={len(bar_lines)}，将只对齐前 {n} 行。")

    out_csv = OUT_DIR / f"bar-de-{pct}.csv"
    with out_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["de", "bar"])
        for i in range(n):
            w.writerow([de_lines[i], bar_lines[i]])

    print(f"Saved -> {out_csv.resolve()} ({n} pairs)")

DE lines: 39
BAR 20 lines: 39
Saved -> /content/drive/MyDrive/data-dass/ablation/aligned-1/bar-de-20.csv (39 pairs)
BAR 40 lines: 39
Saved -> /content/drive/MyDrive/data-dass/ablation/aligned-1/bar-de-40.csv (39 pairs)
BAR 60 lines: 39
Saved -> /content/drive/MyDrive/data-dass/ablation/aligned-1/bar-de-60.csv (39 pairs)
BAR 80 lines: 39
Saved -> /content/drive/MyDrive/data-dass/ablation/aligned-1/bar-de-80.csv (39 pairs)
BAR 100 lines: 39
Saved -> /content/drive/MyDrive/data-dass/ablation/aligned-1/bar-de-100.csv (39 pairs)
